In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from xgboost import XGBClassifier
from sklearn.preprocessing import FunctionTransformer, LabelBinarizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier, BaggingClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline

In [2]:
BASE_DIR = Path.cwd().parent
DATA_DIR = os.path.join(BASE_DIR, "data")
TRAIN_FILE = os.path.join(DATA_DIR, "train_raw.csv")
TEST_FILE = os.path.join(DATA_DIR, "test_raw.csv")
df = pd.read_csv(TRAIN_FILE, index_col=0)
df.head()

,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,gender,marital_status,education_level,employment_status,loan_purpose,grade_subgrade,loan_paid_back
id,,,,,,,,,,,,
0,29367.99,0.084,736,2528.42,13.67,Female,Single,High School,Self-employed,Other,C3,1.0
1,22108.02,0.166,636,4593.10,12.92,Male,Married,Master's,Employed,Debt consolidation,D3,0.0
2,49566.20,0.097,694,17005.15,9.76,Male,Single,High School,Employed,Debt consolidation,C5,1.0
3,46858.25,0.065,533,4682.48,16.10,Female,Single,High School,Employed,Debt consolidation,F1,1.0
4,25496.70,0.053,665,12184.43,10.21,Male,Married,High School,Employed,Other,D1,1.0


In [3]:
# ====Preprocessing====
df_preprocessed = df.copy()
sqrt_transformer = FunctionTransformer(np.sqrt)


one_hot = LabelBinarizer()



In [4]:
train = pd.read_csv(TRAIN_FILE, index_col=0)
test = pd.read_csv(TEST_FILE, index_col=0)

# Target
y_train = train['loan_paid_back']

In [5]:
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder, OrdinalEncoder
from sklearn.pipeline import make_pipeline

drop_cols = ['loan_paid_back', 'employment_status']
train = train.drop(columns=drop_cols)
test = test.drop(columns=drop_cols[1:])

log_features = ['annual_income', 'debt_to_income_ratio']
log_transformer = FunctionTransformer(np.log1p)
log_pipeline = make_pipeline(
    log_transformer,
    StandardScaler()
)

numeric_features = ['credit_score', 'loan_amount', 'interest_rate']
categorical_low = ['gender', 'marital_status', 'education_level', 'loan_purpose']
categorical_high = ['grade_subgrade']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_features),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_low),
        ('freq', OrdinalEncoder(), categorical_high),
        ('log', log_transformer, log_features)
    ])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, val_idx) in enumerate(skf.split(train, y_train)):
    X_tr, X_val = train.iloc[train_idx], train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    X_tr_proc = preprocessor.fit_transform(X_tr)
    X_val_proc = preprocessor.transform(X_val)

    print(f"Fold {fold}: X_tr shape {X_tr_proc.shape}")

Fold 0: X_tr shape (475195, 26)
Fold 1: X_tr shape (475195, 26)
Fold 2: X_tr shape (475195, 26)
Fold 3: X_tr shape (475195, 26)
Fold 4: X_tr shape (475196, 26)


In [7]:
from lightgbm import LGBMClassifier

pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier())
])

search_space = [
    {
        "classifier": [LogisticRegression(max_iter=1000, solver='liblinear', random_state=42)],
        "classifier__l1_ratio": [0, 1],
        "classifier__C": np.logspace(-2, 2, 5)
    },
    {
        "classifier": [RandomForestClassifier(n_jobs=1, random_state=42)],
        "classifier__n_estimators": [100, 300],
        "classifier__max_depth": [5, 10, None],
        "classifier__min_samples_split": [2, 5]
    },
    {
        "classifier": [LGBMClassifier(random_state=42,
                                      verbose=-1,
                                      device_type='gpu',
                                      n_jobs=1)],
        "classifier__n_estimators": [100, 300],
        "classifier__max_depth": [5, 10, -1],
        "classifier__learning_rate": [0.05, 0.1]
    },
    {
        "classifier": [ExtraTreesClassifier(n_jobs=1, random_state=42)],
        "classifier__n_estimators": [100, 300],
        "classifier__max_depth": [5, 10, None]
    },
    {
        "classifier": [XGBClassifier(
            random_state=42,
            n_jobs=1,
            tree_method='gpu_hist',
            gpu_id=0,
            predictor='gpu_predictor'
        )],
        "classifier__n_estimators": [100, 300],
        "classifier__learning_rate": [0.05, 0.1],
        "classifier__max_depth": [6, 10]
    },
]


cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
gridsearch = GridSearchCV(
    pipe, search_space,
    cv=cv,
    scoring='roc_auc',
    n_jobs=2,
    verbose=1
)
best_model = gridsearch.fit(train, y_train)
print(f"Najlepszy model: {best_model.best_estimator_}")
print(f"CV ROC-AUC: {best_model.best_score_:.4f}")
print(f"Ranking CV: {best_model.cv_results_['mean_test_score'][-5:]}")

Fitting 3 folds for each of 48 candidates, totalling 144 fits


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
/home/kaczorekt/PythonProjekty/forecasting-loan-repayment-team-11/venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/kaczorekt/PythonProjekty/forecasting-loan-repayment

Najlepszy model: Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['credit_score',
                                                   'loan_amount',
                                                   'interest_rate']),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['gender', 'marital_status',
                                                   'education_level',
                                                   'loan_purpose']),
                                                 ('freq', OrdinalEncoder(),
                                                  ['grade_subgrade']),
                                                 ('log',
                  